## GPT Classifier
In this pipeline we will classify as relevant or not each element of the test-set

In [ ]:
import sys
from pathlib import Path
# Get parent directory (Thesis-Edvin)
sys.path.append(str(Path.cwd().parent))

In [ ]:
from datasets import load_dataset

test = load_dataset("Eathus/github-issues-vul-detection", split="test")
test_df = test.to_pandas()

### Comparison between Eathus data and DanCip data

In [ ]:
from datasets import load_dataset

test_dan = load_dataset("DanCip/github-issues-vul-detection", split="test")
test_df_dan = test_dan.to_pandas()

In [ ]:
print("len(test_df_dan):", len(test_df_dan))
print("len(test_df):", len(test_df))

print("Dan posetive count:", len(test_df_dan[test_df_dan['label']]))
print("Eathus posetive count:", len(test_df[test_df['label']]))

## Zero-shot learning

In [ ]:
# Cost only of the input
(test_df.issue_msg_n_tokens.sum() / 1000000) * 0.15

In [ ]:
import os
from langchain_groq import ChatGroq  # Groq's LangChain integration
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from utils import *

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

class ReplySchema(BaseModel):
    gpt_description: str = Field(
        description="Describe the vulnerability detected, if any; otherwise, write None"
    )
    gpt_vulnerability: str = Field(
        description="Describe the general type of vulnerability detected, if any; otherwise, write None"
    )
    gpt_confidence: int = Field(
        description="An integer from 1 to 5 indicating your level of confidence  (1 = very low, 2 = low, 3 = medium, 4 = high, 5 = very high)."
    )
    gpt_is_relevant: bool = Field(
        description="Boolean that states if there is a vulnerability"
    )

'''
llm = ChatGroq(
    temperature=0.0,
    model_name="llama3-70b-8192",  # Groq's fastest Llama 3 70B endpoint
    api_key=GROQ_API_KEY     # Replace with your Groq key
)
'''

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0, max_tokens=15500) # maybe set max_token to 14000

prompts_dict = load_prompts(os.getcwd() + "/../utils/prompts")

prompt = ChatPromptTemplate.from_messages(
    [("system", prompts_dict["system_setupd_detailed_desc"]), ("human", "{msg}")],
)


def parser(message: ReplySchema):
    return message.model_dump_json()


llm = llm.with_structured_output(ReplySchema)
chain = prompt | llm | parser

In [ ]:
print(sys.path)

In [ ]:
print(prompt.messages[1])

In [ ]:
import tiktoken
from langchain_core.messages import HumanMessage, SystemMessage

# --- Token counting function ---
def count_tokens(messages, model_name="gpt-4"):
    enc = tiktoken.encoding_for_model(model_name)
    total_tokens = 0
    for msg in messages:
        role_tokens = 4  # system/human/assistant structure adds overhead
        total_tokens += role_tokens + len(enc.encode(msg.content))
    total_tokens += 2  # additional overhead for the assistant priming
    return total_tokens

example_input = "Here's a snippet of Python code to check for vulnerabilities."
filled_prompt = prompt.format_messages(msg=example_input)
# --- Count tokens in the filled prompt ---
token_count = count_tokens(filled_prompt)
print(f"Token count for prompt: {token_count}")

In [ ]:
example = chain.invoke('''This is a GitHub Issue\nrepo: ZipArchive\nowner: ZipArchive\nTitle : Unhandled edge case in _sanitizedPath #680 \n\n--- start body ---\nDescription:\n\nThere is an unhandeled edge case where we can crash the application when it tries to unzip a specially crafted zip file\n\nTechnical details:\n\nBelow is a code snippet from the _sanitizedPath function used to sanitize file paths coming from zip entries before extraction to prevent path traversal, the function prepends file:/// to the provided path, standardizes it using NSURL and then removes the prepended file:///, when provided with the following path however /.., the standardized path returned by NSURL becomes file://, which has 7 characters, the package however expects at least 8 characters (length of the prepended text), this results in crashing the application.\n\n// Add scheme \"file:///\" to support sanitation on names with a colon like \"file:a/../../../usr/bin\"\nstrPath = [@\"file:///\" stringByAppendingString:strPath];\n\n// Sanitize path traversal characters to prevent directory backtracking. Ignoring these characters mimicks the default behavior of the Unarchiving tool on macOS.\n// \"../../../../../../../../../../../tmp/test.txt\" -> \"tmp/test.txt\"\n// \"a/b/../c.txt\" -> \"a/c.txt\"\nstrPath = [NSURL URLWithString:strPath].standardizedURL.absoluteString;\n\n// Remove the \"file:///\" scheme\nstrPath = [strPath substringFromIndex:8];\nPoC:\n\nimport zipfile\n\ndef compress_file(filename):\n    with zipfile.ZipFile('payload.zip', 'w') as zipf:\n        zipf.writestr(filename, \"Test payload\")\n\nfilename = '/..'\n\ncompress_file(filename)\n--- end body ---''')

In [ ]:
import json
import pandas as pd

examplej = json.loads(example)

In [ ]:
from IPython.display import display, Markdown, Latex
display(Markdown(examplej['gpt_vulnerability']))

In [ ]:
from utils import *
from pandarallel import pandarallel
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import OpenAIError, RateLimitError  # Explicitly import errors

pandarallel.initialize(progress_bar=True, nb_workers=15)

@retry(
    stop=stop_after_attempt(5),  # Retry up to 5 times
    wait=wait_exponential(multiplier=2, min=1, max=60),  # Exponential backoff
    retry=retry_if_exception_type(RateLimitError),  # Retry only on rate limit errors
)
def _gpt_classify(msg):
    if not msg or not isinstance(msg, str):  # Check for empty/invalid messages
        return None
    return chain.invoke({"msg": msg})  # Adjusted for OpenAI API format

def gpt_classify(msg):
    try:
        return _gpt_classify(msg)
    except OpenAIError as e:  # Catch all OpenAI-specific errors
        print(f"OpenAI API error: {e}")
    except Exception as e:
        print(f"General error processing message: {e}")
    return None


In [ ]:
import pickle
import os
file_path = "tmp/gpt_response_zero_cwe_df.pkl"

# Define the path to your pickle file

# Check if the file exists
if os.path.exists(file_path):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    print("Pickle file loaded successfully!")
else:
    print(f"The file at {file_path} does not exist. Setting gpt_response to 'None'")
    test_df['gpt_response'] = None



In [ ]:
from datasets import load_dataset

test = load_dataset("Eathus/github-issues-vul-detection-gpt-zero-results", split="test")
test_df = test.to_pandas()

In [ ]:
import json
import pandas as pd

# Step 1: Combine the normalized columns back into a dictionary
test_df['gpt_response'] = test_df.apply(
    lambda row: json.dumps({
        'gpt_description': row['gpt_description'],
        'gpt_confidence': row['gpt_confidence'],
        'gpt_is_relevant': row['gpt_is_relevant']
    }) if pd.notna(row['gpt_description']) else None,
    axis=1
)

# Step 2: Drop the individual normalized columns (optional)
test_df = test_df.drop(columns=['gpt_description', 'gpt_confidence', 'gpt_is_relevant'])

In [ ]:
len(test_df[test_df.gpt_response.isna()])

In [ ]:
import pickle 
import time

retries = 0
max_retries = 10
file_path = "tmp/gpt_response_zero_cwe_df.pkl"
DELAY = 60 
#test_df['gpt_response'] = None
while not test_df[test_df.gpt_response.isna()].empty and retries < max_retries :
    # Get indices of rows needing processing
    na_indices = test_df[test_df.gpt_response.isna()].index
    
    if len(na_indices) == 0:
        break
    
    # Process ONLY those rows and assign directly to original DF
    test_df.loc[na_indices, 'gpt_response'] = \
        test_df.loc[na_indices, 'issue_msg'].parallel_apply(gpt_classify)
    
    with open(file_path, "wb") as file:  # 'wb' mode writes in binary format
        pickle.dump(test_df, file)
    retries += 1
    print(f"Retry {retries}: Processed {len(na_indices)} rows")
    time.sleep(DELAY)

In [ ]:
print(*test_df.sample(3)['gpt_response'])
display(test_df.sample(3)['gpt_response'])

In [ ]:
import json
import pandas as pd

test_df['gpt_response'] = test_df['gpt_response'].map(
    lambda x: json.loads(x) if pd.notna(x) else None
)

In [ ]:
print(*test_df.sample(3)['gpt_response'])
display(type(test_df['gpt_response'][0]))

In [ ]:
import pandas as pd

tmp = pd.json_normalize(test_df["gpt_response"])
test_df = pd.concat([test_df.drop(columns=["gpt_response"]), tmp], axis=1)

In [ ]:
display(test_df.head())

In [ ]:
from datasets import Dataset

# test
ds = Dataset.from_pandas(test_df)
ds.push_to_hub("Eathus/github-issues-vul-detection-gpt-zero-vul-desc-results", split="test")

In [ ]:
test_df[
    test_df.gpt_confidence == 5
].gpt_is_relevant.sum(), test_df.gpt_is_relevant.count()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_test = test_df.label.values
y_pred = test_df.gpt_is_relevant.values

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

## Few-shot learning

In [1]:
import sys
from pathlib import Path

# Get parent directory (Thesis-Edvin)
sys.path.append(str(Path.cwd().parent))

In [2]:
from datasets import load_dataset

test = load_dataset("Eathus/github-issues-vul-detection-max", split="train")
test_df = test.to_pandas()

In [3]:
# Cost only of the input
(test_df.issue_msg_n_tokens.sum() / 1000000) * 0.15

2.98881675

In [11]:
from utils import *
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class ReplySchema(BaseModel):
    gpt_description: str = Field(
        description="Describe the vulnerability detected, if any; otherwise, write None"
    )
    gpt_vulnerability: str = Field(
        description="Describe the general type of vulnerability detected, if any; otherwise, write None"
    )
    gpt_confidence: int = Field(
        description="An integer from 1 to 5 indicating your level of confidence  (1 = very low, 2 = low, 3 = medium, 4 = high, 5 = very high)."
    )
    gpt_is_relevant: bool = Field(
        description="boolean that states if there is a vulnerability"
    )


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    max_retries=10,
    api_key=OPENAI_API_KEY_KTH,
    max_tokens=13500,
)

prompts_dict = load_prompts(os.getcwd() + "/../utils/prompts")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", prompts_dict["system_setupd_detailed_desc_final"]),
        (
            "human",
            """
                aac\nIssue date:\n\n--- start body ---\nHi, developers of Bento4:\r\nIn the test of the binary mp42aac instrumented with ASAN. There are some inputs causing heap-buffer-overflow. Here is the ASAN mode output:\r\n\r\n=================================================================\r\n==4695==ERROR: AddressSanitizer: heap-buffer-overflow on address 0x6190000027a0 at pc 0x7ffff6ef6964 bp 0x7fffffffdea0 sp 0x7fffffffd648\r\nWRITE of size 4294967288 at 0x6190000027a0 thread T0\r\n    #0 0x7ffff6ef6963 in __asan_memcpy (\\/usr\\/lib\\/x86_64-linux-gnu\\/libasan.so.2+0x8c963)\r\n    #1 0x409ed4 in AP4_MemoryByteStream::WritePartial(void const*, unsigned int, unsigned int&) \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Core\\/Ap4ByteStream.cpp:785\r\n    #2 0x40d9e3 in AP4_ByteStream::Write(void const*, unsigned int) \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Core\\/Ap4ByteStream.cpp:77\r\n    #3 0x4eb601 in AP4_Atom::Write(AP4_ByteStream&) \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Core\\/Ap4Atom.cpp:229\r\n    #4 0x4eb601 in AP4_Atom::Clone() \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Core\\/Ap4Atom.cpp:316\r\n    #5 0x446d7a in AP4_SampleDescription::AP4_SampleDescription(AP4_SampleDescription::Type, unsigned int, AP4_AtomParent*) \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Core\\/Ap4SampleDescription.cpp:138\r\n    #6 0x461a8f in AP4_GenericAudioSampleDescription::AP4_GenericAudioSampleDescription(unsigned int, unsigned int, unsigned short, unsigned short, AP4_AtomParent*) \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Core\\/Ap4SampleDescription.h:259\r\n    #7 0x461a8f in AP4_AudioSampleEntry::ToSampleDescription() \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Core\\/Ap4SampleEntry.cpp:630\r\n    #8 0x48ca03 in AP4_StsdAtom::GetSampleDescription(unsigned int) \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Core\\/Ap4StsdAtom.cpp:181\r\n    #9 0x4040b6 in main \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Apps\\/Mp42Aac\\/Mp42Aac.cpp:268\r\n    #10 0x7ffff61bb83f in __libc_start_main (\\/lib\\/x86_64-linux-gnu\\/libc.so.6+0x2083f)\r\n    #11 0x408338 in _start (\\/home\\/ferry\\/dp\\/Bento4\\/mp42aac+0x408338)\r\n\r\n0x6190000027a0 is located 0 bytes to the right of 1056-byte region [0x619000002380,0x6190000027a0)\r\nallocated by thread T0 here:\r\n    #0 0x7ffff6f03712 in operator new[](unsigned long) (\\/usr\\/lib\\/x86_64-linux-gnu\\/libasan.so.2+0x99712)\r\n    #1 0x414c8e in AP4_DataBuffer::ReallocateBuffer(unsigned int) \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Core\\/Ap4DataBuffer.cpp:210\r\n    #2 0x414c8e in AP4_DataBuffer::SetBufferSize(unsigned int) \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Core\\/Ap4DataBuffer.cpp:136\r\n    #3 0x414c8e in AP4_DataBuffer::Reserve(unsigned int) \\/home\\/ferry\\/dp\\/Bento4\\/Source\\/C++\\/Core\\/Ap4DataBuffer.cpp:107\r\n\r\nSUMMARY: AddressSanitizer: heap-buffer-overflow ??:0 __asan_memcpy\r\nShadow bytes around the buggy address:\r\n  0x0c327fff84a0: 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\r\n  0x0c327fff84b0: 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\r\n  0x0c327fff84c0: 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\r\n  0x0c327fff84d0: 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\r\n  0x0c327fff84e0: 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\r\n=>0x0c327fff84f0: 00 00 00 00[fa]fa fa fa fa fa fa fa fa fa fa fa\r\n  0x0c327fff8500: fa fa fa fa fa fa fa fa fa fa fa fa fa fa fa fa\r\n  0x0c327fff8510: fa fa fa fa fa fa fa fa fa fa fa fa fa fa fa fa\r\n  0x0c327fff8520: fa fa fa fa fa fa fa fa fa fa fa fa fa fa fa fa\r\n  0x0c327fff8530: fa fa fa fa fa fa fa fa fa fa fa fa fa fa fa fa\r\n  0x0c327fff8540: fa fa fa fa fa fa fa fa fa fa fa fa fa fa fa fa\r\nShadow byte legend (one shadow byte represents 8 application bytes):\r\n  Addressable:           00\r\n  Partially addressable: 01 02 03 04 05 06 07 \r\n  Heap left redzone:       fa\r\n  Heap right redzone:      fb\r\n  Freed heap region:       fd\r\n  Stack left redzone:      f1\r\n  Stack mid redzone:       f2\r\n  Stack right redzone:     f3\r\n  Stack partial redzone:   f4\r\n  Stack after return:      f5\r\n  Stack use after scope:   f8\r\n  Global redzone:          f9\r\n  Global init order:       f6\r\n  Poisoned by user:        f7\r\n  Container overflow:      fc\r\n  Array cookie:            ac\r\n  Intra object redzone:    bb\r\n  ASan internal:           fe\r\n==4695==ABORTING\r\n\r\n### Crash input\r\nhttps:\\/\\/github.com\\/17ssDP\\/fuzzer_crashes\\/blob\\/main\\/Bento4\\/input2\r\n\r\n### Validation steps\r\n```\r\ngit clone https:\\/\\/github.com\\/axiomatic-systems\\/Bento4\r\ncd Bento4\\/\r\nmkdir check_build && cd check_build\r\ncmake ..\\/ -DCMAKE_C_COMPILER=clang  -DCMAKE_CXX_COMPILER=clang++ -DCMAKE_C_FLAGS=\"-fsanitize=address\" -DCMAKE_CXX_FLAGS=\"-fsanitize=address\" -DCMAKE_BUILD_TYPE=Release\r\nmake -j \r\n.\\/mp42aac input2 \\/dev\\/null\r\n```\r\n### Environment\r\nUbuntu 16.04\r\nClang 10.0.1\r\ngcc 5.5\r\n\r\n\n--- end body ---\n
            """,
        ),
        (
            "ai",
            """
            \"gpt_description\": \"Buffer overflow vulnerability in function AP4_MemoryByteStream::WritePartial in mp42aac in Bento4 v1.6.0-639, allows attackers to cause a denial of service via a crafted file.\", \"gpt_vulnerability\": \"# Buffer Copy without Checking Size of Input (\'Classic Buffer Overflow\')\n\n## Description\nThe product copies an input buffer to an output buffer without verifying that the size of the input buffer is less than the size of the output buffer, leading to a buffer overflow.\n\n## Extended Description\nA buffer overflow condition exists when a product attempts to put more data in a buffer than it can hold, or when it attempts to put data in a memory area outside of the boundaries of a buffer. The simplest type of error, and the most common cause of buffer overflows, is the "classic" case in which the product copies the buffer without restricting how much is copied. Other variants exist, but the existence of a classic overflow strongly suggests that the programmer is not considering even the most basic of security protections.\n\n## Demonstrative Scenario\n\n### Scenario\nThe following code asks the user to enter their last name and then attempts to store the value entered in the last_name array.\n\n#### Vulnerable C Code\n```c\nchar last_name[20];\n\tprintf ("Enter your last name: ");\n\tscanf ("%s", last_name);\n```\n\n#### Analysis\nThe problem with the code above is that it does not restrict or limit the size of the name entered by the user. If the user enters "Very_very_long_last_name" which is 24 characters long, then a buffer overflow will occur since the array can only hold 20 characters total.\", \"gpt_confidence\": 5, \"gpt_is_relevant\": true
            """,
        ),
        #(
        #    "human",
        #    """
        #    This is a GitHub Issue\nrepo:MonetDB\nowner:MonetDB\nTitle : MonetDB server 11.46.0 crashes at list_append #7383\n\nIssue date:\n\n--- start body ---\nDescribe the bug\nMonetDB server 11.46.0 crashes at list_append after executing SQL statements through mclient.\n\nTo Reproduce\n\ncreate table t1(c1 int auto_increment primary key NOT NULL);\ncreate trigger i1 after insert on t1 for each row insert into t1 values(NULL);\ninsert into t1 values(NULL);\nExpected behavior\nThis crash is strange. From the backtrace as follows, we can know that the crash is caused by infinite recursion.\n\nHowever, if we change the first statement into create table t1(c1 int);, the MonetDB will return the message Query too complex: running out of stack space when executing the third stmt. So, MonetDB handles out of stack space, but it does not work with some special table definitions?\n\nWhat's more, the definition of the table contains the NOT NULL constraint, which should fail the third stmt. If we remove the create trigger statement, MonetDB will return the message INSERT INTO: NOT NULL constraint violated for column t1.c1 when executing the insert stmt. But with the create trigger stmt, it crashes instead.\n\nAnyway, I think the expected behavior is to return the error message Query too complex: running out of stack space or INSERT INTO: NOT NULL constraint violated for column t1.c1, instead of crashing straightly.\n\nBacktrace\n\n#0 0x7f0c3760dbda (list_append+0x1a)\n#1 0x7f0c376669c6 (rel_insert+0x196)\n#2 0x7f0c3766abb3 (rel_updates+0x1ce3)\n#3 0x7f0c376c6c11 (sequential_block+0x121)\n#4 0x7f0c376c52dc (rel_psm+0x131c)\n#5 0x7f0c37653091 (rel_semantic+0x91)\n#6 0x7f0c37652e6a (rel_parse+0x19a)\n#7 0x7f0c37563102 (sql_insert_triggers+0x232)\n#8 0x7f0c3755bb5e (rel2bin_insert+0x148e)\n#9 0x7f0c37554539 (subrel_bin+0xd69)\n#10 0x7f0c3755f218 (exp_bin+0x29e8)\n#11 0x7f0c37558567 (subrel_bin+0x4d97)\n#12 0x7f0c37563162 (sql_insert_triggers+0x292)\n#13 0x7f0c3755bb5e (rel2bin_insert+0x148e)\n#14 0x7f0c37554539 (subrel_bin+0xd69)\n#15 0x7f0c3755f218 (exp_bin+0x29e8)\n#16 0x7f0c37558567 (subrel_bin+0x4d97)\n#17 0x7f0c37563162 (sql_insert_triggers+0x292)\n...\n#7407 0x7f0c37563162 (sql_insert_triggers+0x292)\n#7408 0x7f0c3755bb5e (rel2bin_insert+0x148e)\n#7409 0x7f0c37554539 (subrel_bin+0xd69)\n#7410 0x7f0c3755373b (output_rel_bin+0x6b)\n#7411 0x7f0c3757f9d9 (backend_dumpstmt+0x199)\n#7412 0x7f0c3754a367 (SQLparser+0x5d7)\n#7413 0x7f0c3754987b (SQLengine_+0x59b)\n#7414 0x7f0c37548343 (SQLengine+0x23)\n#7415 0x7f0c378d76cf (runScenario+0x4f)\n#7416 0x7f0c378d816c (MSscheduleClient+0x68c)\n#7417 0x7f0c3797fc2b (doChallenge+0xfb)\n#7418 0x7f0c37ffeba0 (THRstarter+0x100)\n#7419 0x7f0c3806ecc4 (thread_starter+0x34)\n#7420 0x7f0c373e3609 (start_thread+0xd9)\n#7421 0x7f0c37308133 (clone+0x43)\nSoftware versions\n\nMonetDB server version: 11.46.0 (hg id: 63a42c2) (pulled from the master branch)\nMonetDB client version: mclient, version 11.48.0 (hg id: 63a42c2)\nOS and version: ubuntu 20.04\nSelf-installed and compiled. The command line of compilation: CC=clang-12 CXX=clang++-12 cmake /root/monetdb_master -DCMAKE_BUILD_TYPE=RelWithDebInfo\nIssue labeling\nbug\n\nAdditional context\nThe MonetDB here runs in-memory database. The command line of starting MonetDB server is mserver5 --in-memory.\n--- end body ---\n
        #    """,
        #),
        #(
        #    "ai",
        #    """
        #    \"gpt_description\": \"An issue in the list_append component of MonetDB Server v11.45.17 and v11.46.0 allows attackers to cause a Denial of Service (DoS) via crafted SQL statements.\", \"gpt_vulnerability\": \"# Allocation of Resources Without Limits or Throttling\n\n## Description\nThe product allocates a reusable resource or group of resources on behalf of an actor without imposing any restrictions on the size or number of resources that can be allocated, in violation of the intended security policy for that actor.\n\n## Extended Description\nCode frequently has to work with limited resources, so programmers must be careful to ensure that resources are not consumed too quickly, or too easily. Without use of quotas, resource limits, or other protection mechanisms, it can be easy for an attacker to consume many resources by rapidly making many requests, or causing larger resources to be used than is needed. When too many resources are allocated, or if a single resource is too large, then it can prevent the code from working correctly, possibly leading to a denial of service.\n\n## Demonstrative Scenario\n\n### Scenario\nThis code allocates a socket and forks each time it receives a new connection.\n\n#### Vulnerable C Code\n```c\nsock=socket(AF_INET, SOCK_STREAM, 0);\n\twhile (1) {{\n\t\tnewsock=accept(sock, ...);\n\t\tprintf("A connection has been accepted\\n");\n\t\tpid = fork();\n\t}}\n```\n\n#### Analysis\nThe program does not track how many connections have been made, and it does not limit the number of connections. Because forking is a relatively expensive operation, an attacker would be able to cause the system to run out of CPU, processes, or memory by making a large number of connections. Alternatively, an attacker could consume all available connections, preventing others from accessing the system remotely.\", \"gpt_confidence\": 5, \"gpt_is_relevant\": true
        #    """,
        #),
        (
            "human",
            """
            This is a GitHub Issue repo:libredwg owner:LibreDWG Title : Segmentation fault in acds_private Issue date: --- start body --- [2.zip](https://github.com/LibreDWG/libredwg/files/11244857/2.zip) ``` (gdb) (gdb) r -v9 /home/user/out/default/crashes/2* The program being debugged has been started already. Start it from the beginning? (y or n) y Starting program: /home/user/libredwg/dwgread -v9 /home/user/out/default/crashes/2* Reading DWG file /home/user/out/default/crashes/2.dwg Program received signal SIGSEGV, Segmentation fault. 0xb6c8eb26 in acds_private (dat=0xbfffe330, dwg=0xbfffe578) at src/acds.spec:215 215 SUB_FIELD_RL (schidx.props[rcount2],index, 0); (gdb) ``` --- end body ---
            """,
        ),
        (
            "ai",
            """
            \"gpt_description\": \"\", \"gpt_vulnerability\": \"\", \"gpt_confidence\": 2,\n    \"gpt_is_relevant\": false
            """,
        ),
        ("human", "{msg}"),
    ]
)


def parser(message: ReplySchema):
    return message.model_dump_json()


llm = llm.with_structured_output(ReplySchema)
chain = prompt | llm | parser

In [ ]:
example = chain.invoke('''This is a GitHub Issue\nrepo: ZipArchive\nowner: ZipArchive\nTitle : Unhandled edge case in _sanitizedPath #680 \n\n--- start body ---\nDescription:\n\nThere is an unhandeled edge case where we can crash the application when it tries to unzip a specially crafted zip file\n\nTechnical details:\n\nBelow is a code snippet from the _sanitizedPath function used to sanitize file paths coming from zip entries before extraction to prevent path traversal, the function prepends file:/// to the provided path, standardizes it using NSURL and then removes the prepended file:///, when provided with the following path however /.., the standardized path returned by NSURL becomes file://, which has 7 characters, the package however expects at least 8 characters (length of the prepended text), this results in crashing the application.\n\n// Add scheme \"file:///\" to support sanitation on names with a colon like \"file:a/../../../usr/bin\"\nstrPath = [@\"file:///\" stringByAppendingString:strPath];\n\n// Sanitize path traversal characters to prevent directory backtracking. Ignoring these characters mimicks the default behavior of the Unarchiving tool on macOS.\n// \"../../../../../../../../../../../tmp/test.txt\" -> \"tmp/test.txt\"\n// \"a/b/../c.txt\" -> \"a/c.txt\"\nstrPath = [NSURL URLWithString:strPath].standardizedURL.absoluteString;\n\n// Remove the \"file:///\" scheme\nstrPath = [strPath substringFromIndex:8];\nPoC:\n\nimport zipfile\n\ndef compress_file(filename):\n    with zipfile.ZipFile('payload.zip', 'w') as zipf:\n        zipf.writestr(filename, \"Test payload\")\n\nfilename = '/..'\n\ncompress_file(filename)\n--- end body ---''')

In [ ]:
import json
import pandas as pd

examplej = json.loads(example)

In [ ]:
from IPython.display import display, Markdown, Latex
display(Markdown(examplej['gpt_vulnerability']))

In [ ]:
example_input = "Here's a snippet of Python code to check for vulnerabilities."
filled_prompt = prompt.format_messages(msg=example_input)
# --- Count tokens in the filled prompt ---
token_count = count_tokens(filled_prompt, 'gpt-4o-mini')
print(f"Token count for prompt: {token_count}")

In [5]:
from utils import *
from pandarallel import pandarallel
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import OpenAIError, RateLimitError  # Explicitly import errors

pandarallel.initialize(progress_bar=True, nb_workers=15)

@retry(
    stop=stop_after_attempt(5),  # Retry up to 5 times
    wait=wait_exponential(multiplier=2, min=1, max=60),  # Exponential backoff
    retry=retry_if_exception_type(RateLimitError),  # Retry only on rate limit errors
)
def _gpt_classify(msg):
    if not msg or not isinstance(msg, str):  # Check for empty/invalid messages
        return None
    return chain.invoke({"msg": msg})  # Adjusted for OpenAI API format

def gpt_classify(msg):
    try:
        return _gpt_classify(msg)
    except OpenAIError as e:  # Catch all OpenAI-specific errors
        print(f"OpenAI API error: {e}")
    except Exception as e:
        print(f"General error processing message: {e}")
    return None


INFO: Pandarallel will run on 15 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [6]:
import pickle
import os
file_path = "tmp/gpt_response_few_cwe_large_train_df.pkl"

# Define the path to your pickle file

# Check if the file exists
if os.path.exists(file_path):
    with open(file_path, 'rb') as file:
        test_df = pickle.load(file)
    print("Pickle file loaded successfully!")
else:
    print(f"The file at {file_path} does not exist. Setting gpt_response to 'None'")
    test_df['gpt_response'] = None



The file at tmp/gpt_response_few_cwe_large_train_df.pkl does not exist. Setting gpt_response to 'None'


In [ ]:
from datasets import load_dataset

test = load_dataset("Eathus/github-issues-vul-detection-gpt-few-results", split="test")
test_df = test.to_pandas()

In [ ]:
import json
import pandas as pd

# Step 1: Combine the normalized columns back into a dictionary
test_df['gpt_response'] = test_df.apply(
    lambda row: json.dumps({
        'gpt_description': row['gpt_description'],
        'gpt_confidence': row['gpt_confidence'],
        'gpt_is_relevant': row['gpt_is_relevant']
    }) if pd.notna(row['gpt_description']) else None,
    axis=1
)

# Step 2: Drop the individual normalized columns (optional)
test_df = test_df.drop(columns=['gpt_description', 'gpt_confidence', 'gpt_is_relevant'])

In [ ]:
len(test_df[test_df.gpt_response.isna()])

In [ ]:
test_df['gpt_response'] = None

In [12]:
import pickle 
import time

retries = 0
max_retries = 10
#file_path = "tmp/gpt_response_few_cwe_train_df.pkl"
DELAY = 1 
#test_df['gpt_response'] = None
while not test_df[test_df.gpt_response.isna()].empty and retries < max_retries :
    # Get indices of rows needing processing
    na_indices = test_df[test_df.gpt_response.isna()].index
    
    if len(na_indices) == 0:
        break
    
    # Process ONLY those rows and assign directly to original DF
    test_df.loc[na_indices, 'gpt_response'] = \
        test_df.loc[na_indices, 'issue_msg'].parallel_apply(gpt_classify)
    
    with open(file_path, "wb") as file:  # 'wb' mode writes in binary format
        pickle.dump(test_df, file)
    retries += 1
    print(f"Retry {retries}: Processed {len(na_indices)} rows")
    time.sleep(DELAY)

OpenAI API error: Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=13500, prompt_tokens=3212, total_tokens=16712, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=2688))
Retry 1: Processed 1 rows


Retry 2: Processed 1 rows


In [ ]:
test_df.sample(3)

In [ ]:
len(test_df[~test_df.gpt_response.isna()])

In [13]:
import json
import pandas as pd

test_df['gpt_response'] = test_df['gpt_response'].map(
    lambda x: json.loads(x) if pd.notna(x) else None
)

In [14]:
import pandas as pd

tmp = pd.json_normalize(test_df["gpt_response"])
test_df = pd.concat([test_df.drop(columns=["gpt_response"]), tmp], axis=1)

In [ ]:
test_df.sample(1)

In [15]:
from datasets import Dataset

# test
ds = Dataset.from_pandas(test_df)
ds.push_to_hub("Eathus/github-issues-vul-max-detection-gpt-few-results-final3", split="train")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Eathus/github-issues-vul-max-detection-gpt-few-results-final3/commit/3e51b14e546e8b3211f993370e33a27f71a12d3e', commit_message='Upload dataset', commit_description='', oid='3e51b14e546e8b3211f993370e33a27f71a12d3e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Eathus/github-issues-vul-max-detection-gpt-few-results-final3', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Eathus/github-issues-vul-max-detection-gpt-few-results-final3'), pr_revision=None, pr_num=None)

In [ ]:
pd.set_option('display.max_colwidth', None)
print(test_df.gpt_vulnerability.sample(1))

In [ ]:
test_df[~test_df.issue_msg.isna()]

In [ ]:
display(Markdown('''# Improper Limitation of a Pathname to a Restricted Directory ('Path Traversal')

## Summary
The product does not properly validate input, allowing an attacker to access memory outside of the intended bounds, leading to a segmentation fault.

## Extended Description
Segmentation faults occur when a program attempts to read or write to a memory location that it is not allowed to access. This can happen due to various reasons, including improper handling of user input, such as array indices that exceed the allocated size. In this case, the JavaScript code provided as input causes the interpreter to attempt to access an invalid memory location, which can lead to crashes or other unintended behavior. This type of vulnerability can be exploited by attackers to cause denial of service or potentially execute arbitrary code if they can control the input.

## Demonstrative Example

### Intro
The following code attempts to access an array element based on user input without validating the index.

### Vulnerable Code
```c
int arr[10];
int index = get_user_input(); // Assume this function gets an index from user input
if (index < 0 || index >= 10) {
    // No validation, potential out-of-bounds access
}
int value = arr[index]; // This line can cause a segmentation fault if index is out of bounds
```

### Analysis
In the example above, if the user provides an index that is less than 0 or greater than or equal to 10, the program will attempt to access memory outside the bounds of the array, leading to a segmentation fault. This is similar to the issue described in the GitHub report, where the JavaScript input causes the interpreter to access invalid memory, resulting in a crash.'''))

## Results

In [ ]:
from datasets import load_dataset

test = load_dataset(
    "Eathus/github-issues-vul-detection-gpt-few-strict-vul-desc-results", split="test"
)
test_df = test.to_pandas()

In [ ]:
test_df[
    test_df.gpt_confidence == 5
].gpt_is_relevant.sum(), test_df.gpt_is_relevant.count()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_test = test_df.label.values
y_pred = test_df.gpt_is_relevant.values

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print("Classification Report:")
print(report)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot(cmap="Blues")

In [ ]:
test_df.gpt_confidence.describe()